# Sentiment Analysis

### Importing Libraries

In [13]:
import tensorflow as tf
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
import re

### Importing Data & Splitting 

In [14]:
data = pd.read_csv('../Data/sentiment_analysis.csv', index_col= False)
nlp = spacy.load("en_core_web_sm")

y = data['sentiment']
X = data.drop(columns= ['sentiment'])

train, temp, y_train, y_temp = train_test_split(
    X, y, test_size= 0.3, random_state= 42, stratify= y
)

test, validation, y_test, y_validation = train_test_split(
    temp, y_temp, test_size= 0.5, random_state= 42, stratify= y_temp
)

### Pre-Processing Data

In [15]:
# lowercased the whole text
def lowercase(df):
    df = df.copy()
    df['text'] = df['text'].str.lower()
    return df

# removed Urls from the text
def remove_urls(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'https\S+|www\S+', '', x))
    return df

# removed numbers from the text
def remove_numbers(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'\d+', '', x))
    return df

# handling stopwords with removing negation terms
negations = {'not', 'no', 'nor', 'never', "n't", "cannot", "none", "neither"}
stopwords = nlp.Defaults.stop_words - negations

def remove_stopwords(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if t.text not in stopwords]))
    return df

# lemmatize 
def lemmatize(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: ' '.join([t.lemma_ for t in nlp(x)]))
    return df

# punctuation removal 
def remove_punctuation(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if not t.is_punct]))
    return df

def clean_text(df):
    df = df.copy()
    df = lowercase(df)
    df = remove_urls(df)
    df = remove_numbers(df)
    df = remove_stopwords(df)
    df = lemmatize(df)
    df = remove_punctuation(df)
    return df

train = clean_text(train)
test = clean_text(test)
validation = clean_text(validation)

In [16]:
train

,Year,Month,Day,Time of Tweet,text,Platform
233,2023,1,31,night,yes work,Facebook
276,2020,1,5,morning,happy mother day,Facebook
229,2023,9,4,noon,fun night listen episode turkey drama,Instagram
325,2023,2,13,noon,sore throat plan tet outing marwell good time,Twitter
364,2020,8,12,noon,hungry twitter want food,Twitter
...,...,...,...,...,...,...
191,2023,1,20,morning,that`s awesome dude yay surprise celebrity...,Twitter
231,2023,1,15,morning,screw review think wolverine awesome not domin...,Twitter
178,2023,5,30,night,smelly noooo love alex vixon,Instagram
88,2019,3,2,noon,clean house family comme later today,Instagram
